In [4]:
#LIBRERIAS
%pip install ib_async

Using cached ib_async-2.1.0-py3-none-any.whl (88 kB)

   ---------------------------------------- 0/2 [aeventkit]
   -------------------- ------------------- 1/2 [ib_async]
   -------------------- ------------------- 1/2 [ib_async]
   -------------------- ------------------- 1/2 [ib_async]
   ---------------------------------------- 2/2 [ib_async]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# AZUCAR - ACTUALIZACIÓN DE PRECIOS DESDE INTERACTIVE BROKERS
from ib_async import IB, Future
from pathlib import Path
from datetime import datetime, timedelta, date
import pandas as pd
import json, re, os, tempfile
import asyncio

# ========= CONFIG =========
HOST = "127.0.0.1"
PORT = 7496        # 7497 (paper) / 7496 (live)
CLIENT_ID = 16

JSON_PATH = Path(r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_sb.json")

# FECHA MÍNIMA - Solo traer datos desde esta fecha
START_DATE = "2025-01-01"

ONE_DIGIT_YEAR_2020s = False
# ==========================

MONTH_LETTERS = {'H': '03', 'K': '05', 'N': '07', 'V': '10'}
MONTH_NUM_TO_LETTER = {v: k for k, v in MONTH_LETTERS.items()}
ORDERED_MONTHS = ['H', 'K', 'N', 'V']

# ---------- util contrato ----------
def sb_code_to_yyyymm(code: str) -> str:
    m = re.fullmatch(r"SB([HKNV])(\d{1,2})", code.strip().upper())
    if not m:
        raise ValueError(f"Código inválido SB: {code}")
    letter, yy = m.groups()
    if len(yy) == 1:
        year = 2020 + int(yy)
    else:
        yy_i = int(yy)
        year = 2000 + yy_i if yy_i <= 79 else 1900 + yy_i
    return f"{year}{MONTH_LETTERS[letter]}"

def yyyymm_to_sb_code(yyyymm: str) -> str:
    y = int(yyyymm[:4])
    mm = yyyymm[4:6]
    letter = MONTH_NUM_TO_LETTER[mm]
    yy2 = str(y)[-2:]
    if ONE_DIGIT_YEAR_2020s and (2020 <= y <= 2029):
        return f"SB{letter}{y % 10}"
    return f"SB{letter}{yy2}"

async def qualify_sb_async(ib: IB, yyyymm: str):
    for ex in ("NYBOT", "ICEUS"):
        fut = Future(
            symbol="SB",
            lastTradeDateOrContractMonth=yyyymm,
            exchange=ex,
            currency="USD",
            includeExpired=True
        )
        try:
            qs = await ib.qualifyContractsAsync(fut)
            if qs:
                return qs[0]
        except Exception:
            pass
        await asyncio.sleep(0.05)
    return None

# ---------- históricos ----------
async def fetch_history_from_date(ib: IB, q, start_date: str):
    """Descarga histórico desde start_date hasta hoy"""
    start_dt = pd.to_datetime(start_date)
    end_dt = datetime.now()
    days = max(1, (end_dt - start_dt).days + 1)
    
    for what in ("TRADES", "BID_ASK", "MIDPOINT"):
        try:
            bars = await ib.reqHistoricalDataAsync(
                q, endDateTime="", durationStr=f"{days} D",
                barSizeSetting="1 day", whatToShow=what,
                useRTH=False, formatDate=2
            )
            if bars:
                rows = []
                for b in bars:
                    d = str(b.date)
                    if d >= start_date:  # Solo fechas >= START_DATE
                        rows.append({
                            "date": d, "open": b.open, "high": b.high, "low": b.low,
                            "close": b.close, "volume": b.volume, "openinterest": None
                        })
                if rows:
                    df = pd.DataFrame(rows).drop_duplicates("date").sort_values("date")
                    return df.to_dict(orient="records")
        except Exception:
            pass
        await asyncio.sleep(0.25)
    return []

# ---------- JSON I/O ----------
def load_db(path: Path) -> dict:
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def atomic_save(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=path.name, dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        os.replace(tmp, path)
    except Exception:
        try:
            os.remove(tmp)
        finally:
            raise

# ---------- main async ----------
async def main_async():
    db = load_db(JSON_PATH)
    
    ib = IB()
    try:
        await ib.connectAsync(HOST, PORT, clientId=CLIENT_ID)
    except Exception as e:
        print("No se pudo conectar a IBKR:", e)
        return

    try:
        today = datetime.now()
        
        # Contratos existentes en JSON
        sb_contracts = [k for k in db.keys() if re.fullmatch(r"SB[HKNV]\d{1,2}", k)]
        print(f"Contratos en JSON: {len(sb_contracts)}")
        print(f"Fecha mínima: {START_DATE}\n")
        
        updated = 0
        
        for code in sorted(sb_contracts):
            try:
                yyyymm = sb_code_to_yyyymm(code)
            except ValueError:
                continue
            
            q = await qualify_sb_async(ib, yyyymm)
            if not q:
                print(f"⚠️ {code}: no calificado en IBKR")
                continue
            
            # Obtener última fecha en JSON
            bars = db.get(code, [])
            # Filtrar datos anteriores a START_DATE
            bars = [r for r in bars if r["date"] >= START_DATE]
            last_date = max((r["date"] for r in bars), default=None)
            
            if last_date:
                # Ya tiene datos, buscar desde última fecha + 1
                start = (pd.to_datetime(last_date) + timedelta(days=1)).strftime("%Y-%m-%d")
                if start <= today.strftime("%Y-%m-%d"):
                    new_rows = await fetch_history_from_date(ib, q, start)
                    if new_rows:
                        merged = (pd.concat([pd.DataFrame(bars), pd.DataFrame(new_rows)], ignore_index=True)
                                  .drop_duplicates("date").sort_values("date"))
                        db[code] = merged.to_dict(orient="records")
                        print(f"✅ {code}: +{len(new_rows)} filas (hasta {merged.iloc[-1]['date']})")
                        updated += 1
                    else:
                        print(f"   {code}: al día ({last_date})")
                else:
                    print(f"   {code}: al día ({last_date})")
            else:
                # Sin datos, descargar desde START_DATE
                rows = await fetch_history_from_date(ib, q, START_DATE)
                if rows:
                    db[code] = rows
                    print(f"✅ {code}: {len(rows)} filas (desde {START_DATE})")
                    updated += 1
                else:
                    print(f"⚠️ {code}: sin datos")
            
            await asyncio.sleep(0.5)
        
        atomic_save(JSON_PATH, db)
        
        print(f"\n{'='*50}")
        print(f"Actualizados: {updated} | Archivo: {JSON_PATH.name}")
        print(f"{'='*50}")
        
        print("\nÚltima fecha por contrato:")
        for code in sorted(sb_contracts, key=lambda x: sb_code_to_yyyymm(x)):
            bars = db.get(code, [])
            last = max((r["date"] for r in bars), default="N/A")
            print(f"  {code}: {last}")

    finally:
        if ib.isConnected():
            ib.disconnect()

await main_async()

Contratos en JSON: 24
Fecha mínima: 2025-01-01

✅ SBH26: +8 filas (hasta 2026-02-27)
✅ SBH27: +10 filas (hasta 2026-03-03)


Error 200, reqId 7: No se encuentra definici\u00f3n del activo solicitado, contract: Future(symbol='SB', lastTradeDateOrContractMonth='202403', exchange='NYBOT', currency='USD', includeExpired=True)
Unknown contract: Future(symbol='SB', lastTradeDateOrContractMonth='202403', exchange='NYBOT', currency='USD', includeExpired=True)


⚠️ SBH4: no calificado en IBKR


reqHistoricalData: Timeout for Future(conId=554291367, symbol='SB', lastTradeDateOrContractMonth='20250228', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBH5', tradingClass='SB')
Error 366, reqId 9: No se encontraron datos hist\u00f3ricos para ticker id:9, contract: Future(conId=554291367, symbol='SB', lastTradeDateOrContractMonth='20250228', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBH5', tradingClass='SB')
reqHistoricalData: Timeout for Future(conId=554291367, symbol='SB', lastTradeDateOrContractMonth='20250228', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBH5', tradingClass='SB')
Error 366, reqId 10: No se encontraron datos hist\u00f3ricos para ticker id:10, contract: Future(conId=554291367, symbol='SB', lastTradeDateOrContractMonth='20250228', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBH5', tradingClass='SB')
reqHistoricalData: Timeout for Future(conId=554291367, symbol='SB', last

   SBH5: al día (2025-02-28)
✅ SBH6: +8 filas (hasta 2026-02-27)
✅ SBH7: +10 filas (hasta 2026-03-03)
✅ SBH8: +10 filas (hasta 2026-03-03)
✅ SBK26: +10 filas (hasta 2026-03-03)


...

⚠️ SBK4: sin datos
   SBK5: al día (2025-04-30)
✅ SBK6: +10 filas (hasta 2026-03-03)
✅ SBK7: +10 filas (hasta 2026-03-03)
✅ SBK8: +10 filas (hasta 2026-03-03)
✅ SBN26: +10 filas (hasta 2026-03-03)


reqHistoricalData: Timeout for Future(conId=506216909, symbol='SB', lastTradeDateOrContractMonth='20240628', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBN4', tradingClass='SB')
Error 366, reqId 37: No se encontraron datos hist\u00f3ricos para ticker id:37, contract: Future(conId=506216909, symbol='SB', lastTradeDateOrContractMonth='20240628', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBN4', tradingClass='SB')
reqHistoricalData: Timeout for Future(conId=506216909, symbol='SB', lastTradeDateOrContractMonth='20240628', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBN4', tradingClass='SB')
Error 366, reqId 38: No se encontraron datos hist\u00f3ricos para ticker id:38, contract: Future(conId=506216909, symbol='SB', lastTradeDateOrContractMonth='20240628', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBN4', tradingClass='SB')
reqHistoricalData: Timeout for Future(conId=506216909, symbol='SB', la

⚠️ SBN4: sin datos
   SBN5: al día (2025-07-01)
✅ SBN6: +10 filas (hasta 2026-03-03)
✅ SBN7: +10 filas (hasta 2026-03-03)
✅ SBN8: +10 filas (hasta 2026-03-03)
✅ SBV26: +10 filas (hasta 2026-03-03)


reqHistoricalData: Timeout for Future(conId=524270532, symbol='SB', lastTradeDateOrContractMonth='20240930', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBV4', tradingClass='SB')
Error 366, reqId 53: No se encontraron datos hist\u00f3ricos para ticker id:53, contract: Future(conId=524270532, symbol='SB', lastTradeDateOrContractMonth='20240930', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBV4', tradingClass='SB')
reqHistoricalData: Timeout for Future(conId=524270532, symbol='SB', lastTradeDateOrContractMonth='20240930', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBV4', tradingClass='SB')
Error 366, reqId 54: No se encontraron datos hist\u00f3ricos para ticker id:54, contract: Future(conId=524270532, symbol='SB', lastTradeDateOrContractMonth='20240930', multiplier='112000', exchange='NYBOT', currency='USD', localSymbol='SBV4', tradingClass='SB')
reqHistoricalData: Timeout for Future(conId=524270532, symbol='SB', la

⚠️ SBV4: sin datos
   SBV5: al día (2025-10-01)
✅ SBV6: +10 filas (hasta 2026-03-03)
✅ SBV7: +10 filas (hasta 2026-03-03)

Actualizados: 16 | Archivo: data_sb.json

Última fecha por contrato:
  SBH4: 2024-02-29
  SBK4: 2024-04-30
  SBN4: 2024-06-28
  SBV4: 2024-09-30
  SBH5: 2025-02-28
  SBK5: 2025-04-30
  SBN5: 2025-07-01
  SBV5: 2025-10-01
  SBH6: 2026-02-27
  SBH26: 2026-02-27
  SBK6: 2026-03-03
  SBK26: 2026-03-03
  SBN6: 2026-03-03
  SBN26: 2026-03-03
  SBV6: 2026-03-03
  SBV26: 2026-03-03
  SBH7: 2026-03-03
  SBH27: 2026-03-03
  SBK7: 2026-03-03
  SBN7: 2026-03-03
  SBV7: 2026-03-03
  SBH8: 2026-03-03
  SBK8: 2026-03-03
  SBN8: 2026-03-03


In [2]:
# PRECIOS DEL ÚLTIMO DÍA DEL MES ANTERIOR - TODOS LOS FUTUROS
from datetime import date
from dateutil.relativedelta import relativedelta
import pandas as pd
import json
from pathlib import Path

JSON_PATH = Path(r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_sb.json")

# Cargar datos
with open(JSON_PATH, "r", encoding="utf-8") as f:
    db = json.load(f)

# Calcular mes anterior
hoy = date.today()
primer_dia_mes_actual = hoy.replace(day=1)
ultimo_dia_mes_anterior = primer_dia_mes_actual - relativedelta(days=1)
mes_anterior = ultimo_dia_mes_anterior.strftime("%Y-%m")

print(f"Buscando precios del mes: {mes_anterior}")
print(f"(Último día del mes anterior: {ultimo_dia_mes_anterior})\n")

# Recopilar precios del último día del mes anterior para cada contrato
resultados = []

import re
sb_contracts = [k for k in db.keys() if re.fullmatch(r"SB[HKNV]\d{1,2}", k)]

for code in sorted(sb_contracts):
    bars = db.get(code, [])
    
    # Filtrar solo fechas del mes anterior
    fechas_mes = [r for r in bars if r["date"].startswith(mes_anterior)]
    
    if fechas_mes:
        # Obtener el último día disponible del mes
        ultimo_registro = max(fechas_mes, key=lambda x: x["date"])
        resultados.append({
            "Contrato": code,
            "Fecha": ultimo_registro["date"],
            "Close": ultimo_registro["close"],
            "Volume": ultimo_registro["volume"]
        })
    else:
        resultados.append({
            "Contrato": code,
            "Fecha": "Sin datos",
            "Close": None,
            "Volume": None
        })

# Crear DataFrame
df_ultimo_dia_mes = pd.DataFrame(resultados)
print(f"{'='*60}")
print(f"PRECIOS ÚLTIMO DÍA DEL MES ANTERIOR ({mes_anterior})")
print(f"{'='*60}")
display(df_ultimo_dia_mes)

Buscando precios del mes: 2026-02
(Último día del mes anterior: 2026-02-28)

PRECIOS ÚLTIMO DÍA DEL MES ANTERIOR (2026-02)


,Contrato,Fecha,Close,Volume
0,SBH26,2026-02-27,14.30,1803.0
1,SBH27,2026-02-27,14.89,4712.0
2,SBH4,Sin datos,NaN,NaN
3,SBH5,Sin datos,NaN,NaN
4,SBH6,2026-02-27,14.30,1803.0
5,SBH7,2026-02-27,14.89,4712.0
6,SBH8,2026-02-27,15.61,27.0
7,SBK26,2026-02-27,13.89,40103.0
8,SBK4,Sin datos,NaN,NaN
9,SBK5,Sin datos,NaN,NaN
